**This assignment accomplishes the following objectives:**
1. Learn how to invoke the OpenAI and Groq APIs
2. Interpret usage statistics (tokens and latency)
3. Recognize the limitations of standalone LLMs
4. Understand the difference between an LLM and an application built on top of it
5. Compare inference performance across providers

**In order to complete the assignmet:**
1. Work through each cell - pay attention to the code and comments.
2. Briefly answer all eight questions (Q1-Q8) inline as you work through each cell.

**To submit:**
1. Rename the assignment as Assignment1-<asurite>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas





In [ ]:
# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq

In [ ]:
from google.colab import userdata
import os
import aisuite

# read the API_KEYs from Colab Secrets and expose it as an env variable
# IMPORTANT: Make sure to add your OPENAI_API_KEY and GROQ_API_KEY to Colab Secrets
# Click on the '🔑' icon (Secrets) in the left sidebar to add them.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# create the client
aisuite_client = aisuite.Client()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL

In [ ]:
# used for wall-clock timing for request
import time

def run_openai_query(system_prompt, user_prompt):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # display response
  response = completion.choices[0].message.content.strip()
  print(f"\n--- RESPONSE ---")
  print(response)

  # display usage stats
  usage = completion.usage
  print(f"\n--- USAGE STATS ---")
  print(f"• Prompt Tokens:  {usage.prompt_tokens}")
  print(f"• Completion Tokens: {usage.completion_tokens}")
  print(f"• Total Tokens:  {usage.total_tokens}")
  if hasattr(usage, 'total_time'):
    print(f"• Total Time:  {usage.total_time:.2f}s")
  else:
    print(f"• Total Time:  {elapsed_time:.2f}s")


In [ ]:
# used for wall-clock timing for request
import time

def run_groq_query(system_prompt, user_prompt):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to Groq-backend model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_GROQ_MODEL, # Groq-backend model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # display response
  response = completion.choices[0].message.content.strip()
  print(f"\n--- RESPONSE ---")
  print(response)

  # display usage stats
  usage = completion.usage
  print(f"\n--- USAGE STATS ---")
  print(f"• Prompt Tokens:  {usage.prompt_tokens}")
  print(f"• Completion Tokens: {usage.completion_tokens}")
  print(f"• Total Tokens:  {usage.total_tokens}")
  if hasattr(usage, 'total_time'):
    print(f"• Total Time:  {usage.total_time:.2f}s")
  else:
    print(f"• Total Time:  {elapsed_time:.2f}s")


In [ ]:
# Our first OpenAI API call - say hello!
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "Hello!"
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
Hello! How can I assist you today?

--- USAGE STATS ---
• Prompt Tokens:  56
• Completion Tokens: 9
• Total Tokens:  65
• Total Time:  1.85s


In [ ]:
# Let's check the knowledge cutoff date on this model:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "When were you last updated?"
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
My knowledge was last updated in June 2024.

--- USAGE STATS ---
• Prompt Tokens:  60
• Completion Tokens: 11
• Total Tokens:  71
• Total Time:  0.63s


In [ ]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q1) What response did you get?
#  Q2) How is the ChatGPT app able to provide current information?
##############################################################

Q1) It says ChatGPT's knowledge was last updated in June 2024, and it shows prompt tokens, completion tokens, total tokens and total time.

Q2) By pulling data and information from June 2024.

In [ ]:
# Ask for the current time:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "What is the current time?"
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
I don't have real-time capabilities to provide the current time. Please check your device's clock or use an online time service.

--- USAGE STATS ---
• Prompt Tokens:  60
• Completion Tokens: 25
• Total Tokens:  85
• Total Time:  0.75s


In [ ]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q3) What response did you get?
#  Q4) Why do you think we're getting the answer we are?
##############################################################

Q3) "I don't have real-time capabilities to provide the current time. Please check your device's clock or use an online time service."

I figured that ChatGPT can't provide real-time capabilities to display current time.

Q4) I think it is because ChatGPT's information update is stuck in June 2024.

In [ ]:
# Let's ask it about a current event:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "Tell me about the Netflix Warner Bros acquisition."
)
run_openai_query(system_prompt, user_prompt)


--- RESPONSE ---
As of June 2024, there has been no confirmed acquisition of Warner Bros by Netflix. Warner Bros remains a part of Warner Bros. Discovery, a major entertainment company formed by the 2022 merger of WarnerMedia and Discovery, Inc. While Netflix continues to be a leading streaming service and pursues content partnerships and original production, it has not announced or completed any merger or acquisition involving Warner Bros.

There have been industry rumors and speculative discussions about potential consolidation among major studios and streaming platforms amid intense competition, but no official deal has materialized between Netflix and Warner Bros. For verified updates, refer to official press releases from Netflix and Warner Bros. Discovery or trusted financial news sources.

--- USAGE STATS ---
• Prompt Tokens:  63
• Completion Tokens: 137
• Total Tokens:  200
• Total Time:  2.46s


In [ ]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q5) What response did you get?
#  Q6) How is the ChatGPT app able to provide current information?
##############################################################

Q5) This is the exact quote-on-quote answer that I've got.

"As of June 2024, there has been no confirmed acquisition of Warner Bros by Netflix. Warner Bros remains a part of Warner Bros. Discovery, a major entertainment company formed by the 2022 merger of WarnerMedia and Discovery, Inc. While Netflix continues to be a leading streaming service and pursues content partnerships and original production, it has not announced or completed any merger or acquisition involving Warner Bros.

There have been industry rumors and speculative discussions about potential consolidation among major studios and streaming platforms amid intense competition, but no official deal has materialized between Netflix and Warner Bros. For verified updates, refer to official press releases from Netflix and Warner Bros. Discovery or trusted financial news sources."

Basically Warner Bro's has never become a part of Netflix (As of June 2024).

Q6) If we ask chatgpt about a current event (referring to a date after its last training date), it will do a current web search. However, if we ask about the event happened befofe, it will provide pre-trained stuff.

In [ ]:
# Let's compare OpenAI to Groq on a task:
system_prompt = (
    "Provide a high-density, informative response. Avoid filler. "
    "Do not be overly brief, but do not be verbose. Aim for 'the perfect spot': "
    "comprehensive enough to answer fully, but concise enough to read quickly."
)
user_prompt = (
    "Output the integers from 1 to 3000, in a comma-separated list. "
    "Do not include any extra spaces or text."
)

print ("\n############ OpenAI Query ############")
run_openai_query(system_prompt, user_prompt)

print ("\n############ Groq Query ############")
run_groq_query(system_prompt, user_prompt)


############ OpenAI Query ############

--- RESPONSE ---
1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,26

In [ ]:
##############################################################
# Based on above cell's output, answer the following questions:
#  Q7) Why did we pick this task to compare OpenAI to Groq?
#  Q8) Can you explain the difference in usage statistics?
##############################################################

Q7) To see if there's a task execution difference between the token usage and proessing speed of Groq and OpenAI.

Q8) In simple terms, Groq is faster and more costly.

Token counts: OpenAI: 80 prompt / 8000 completion (8080 total); Groq: 104 prompt / 8001 completion (8105 total). The minor discrepancies arise from distinct tokenization methods in their model stacks.

Time: OpenAI took 109.90s vs. Groq's 12.14s—a ~9x speedup for Groq, driven by its LPU hardware optimized for high-throughput inference on long outputs.